In [1]:
import requests
import polars as pl

url = "https://api.open-meteo.com/v1/forecast"

lugares = {
    'Town': ['Berriz', 'Madrid', 'Barcelona'],
    'Latitude': [43.1759, 40.4168, 41.385],
    'Longitude': [-2.5768, -3.7038, 2.1686]
}

all_dfs = []

for town, lat, lon in zip(lugares["Town"], lugares["Latitude"], lugares["Longitude"]):

    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": "temperature_2m,wind_speed_10m,relative_humidity_2m",
        "timezone": "Europe/Madrid"
    }

    data = requests.get(url, params=params).json()

    df = pl.DataFrame({
        "town": [town] * len(data["hourly"]["time"]),
        "time": data["hourly"]["time"],
        "temp": data["hourly"]["temperature_2m"],
        "wind": data["hourly"]["wind_speed_10m"],
        "humidity": data["hourly"]["relative_humidity_2m"],
    })

    all_dfs.append(df)

df_final = pl.concat(all_dfs)
df_final

town,time,temp,wind,humidity
str,str,f64,f64,i64
"""Berriz""","""2026-05-29T00:00""",18.0,4.3,93
"""Berriz""","""2026-05-29T01:00""",16.5,1.1,100
"""Berriz""","""2026-05-29T02:00""",15.8,4.5,100
"""Berriz""","""2026-05-29T03:00""",15.8,6.6,100
"""Berriz""","""2026-05-29T04:00""",15.2,0.5,100
…,…,…,…,…
"""Barcelona""","""2026-06-04T19:00""",24.0,9.0,74
"""Barcelona""","""2026-06-04T20:00""",23.6,5.9,76
"""Barcelona""","""2026-06-04T21:00""",23.1,3.1,80


In [2]:
import requests
import pandas as pd
from pandas_gbq import to_gbq

class tablaAPI:
    def __init__(self):
        self.url = "https://api.open-meteo.com/v1/forecast"
        self.lugares = {
            'Town': ['Berriz', 'Madrid', 'Barcelona'],
            'Latitude': [43.1759, 40.4168, 41.385],
            'Longitude': [-2.5768, -3.7038, 2.1686]
        }
    def datos(self):
        all_dfs = []

        for town, lat, lon in zip(
            self.lugares["Town"],
            self.lugares["Latitude"],
            self.lugares["Longitude"]
        ):
            params = {
                "latitude": lat,
                "longitude": lon,
                "hourly": "temperature_2m,wind_speed_10m,relative_humidity_2m",
                "timezone": "Europe/Madrid"
            }

            data = requests.get(self.url, params=params).json()

            df = pd.DataFrame({
                "town": [town] * len(data["hourly"]["time"]),
                "time": data["hourly"]["time"],
                "temp": data["hourly"]["temperature_2m"],
                "wind": data["hourly"]["wind_speed_10m"],
                "humidity": data["hourly"]["relative_humidity_2m"]
            })

            all_dfs.append(df)

        return pd.concat(all_dfs)

class BigQueryLoader:
    def __init__(self, project_id, table_name):
        self.table_name = table_name
        self.project_id = project_id
    def loader(self, df):
        to_gbq(
            df,
            destination_table = self.table_name,
            project_id = self.project_id,
            if_exists = 'replace'
        )

In [ ]:
ctx = pl.SQLContext()
ctx.register("my_table", df_final)
result = ctx.execute(
    """
    with datos_dia as (
    select
        town,
        cast(
            cast(time as datetime) as date
        ) as date,
        temp,
        wind,
        humidity
    from my_table
    ),
    agregado_lugar_dia as (
    select
        town,
        date,
        max(temp) as max_temp
    from datos_dia
    group by town, date
    ),
    dia_anterior AS (
    select
        town,
        date,
        max_temp,
        lag(max_temp) over (
            partition by town
            order by date
        ) AS max_anterior
    from agregado_lugar_dia
    )
    select
        town,
        date,
        coalesce(max_temp - max_anterior, 0) as diferencia_temperatura_max
    from dia_anterior
    order by town, date
    """,
    eager = True
)

In [4]:
ctx = pl.SQLContext()
ctx.register("my_table", df_final)
result = ctx.execute(
    """
    select * from my_table
    """,
    eager = True
)
print(result)

shape: (504, 5)
┌───────────┬──────────────────┬──────┬──────┬──────────┐
│ town      ┆ time             ┆ temp ┆ wind ┆ humidity │
│ ---       ┆ ---              ┆ ---  ┆ ---  ┆ ---      │
│ str       ┆ str              ┆ f64  ┆ f64  ┆ i64      │
╞═══════════╪══════════════════╪══════╪══════╪══════════╡
│ Berriz    ┆ 2026-05-29T00:00 ┆ 18.0 ┆ 4.3  ┆ 93       │
│ Berriz    ┆ 2026-05-29T01:00 ┆ 16.5 ┆ 1.1  ┆ 100      │
│ Berriz    ┆ 2026-05-29T02:00 ┆ 15.8 ┆ 4.5  ┆ 100      │
│ Berriz    ┆ 2026-05-29T03:00 ┆ 15.8 ┆ 6.6  ┆ 100      │
│ Berriz    ┆ 2026-05-29T04:00 ┆ 15.2 ┆ 0.5  ┆ 100      │
│ …         ┆ …                ┆ …    ┆ …    ┆ …        │
│ Barcelona ┆ 2026-06-04T19:00 ┆ 24.0 ┆ 9.0  ┆ 74       │
│ Barcelona ┆ 2026-06-04T20:00 ┆ 23.6 ┆ 5.9  ┆ 76       │
│ Barcelona ┆ 2026-06-04T21:00 ┆ 23.1 ┆ 3.1  ┆ 80       │
│ Barcelona ┆ 2026-06-04T22:00 ┆ 22.4 ┆ 0.5  ┆ 84       │
│ Barcelona ┆ 2026-06-04T23:00 ┆ 21.6 ┆ 2.9  ┆ 87       │
└───────────┴──────────────────┴──────┴──────┴──────────